## Computational Physics -- Problem Set 6
##### _Theodore Landa_

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import pickle
import matplotlib.pyplot as plt
from scipy import optimize
from math import factorial
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

### Problem 1.

_(a)_ Plot!

_(b)_ For my ok guess, which is `pars = [1, 1.1, 13.3]`, I get a loss of `0.18948935`.

_(c)_ First, I wrote my own gradient descent, which gave me `pars = [ 0.9939996  1.0031334 13.703345 ]` with associated loss `0.019214846`. Then, I tried scipy.optimize (which used L-BFGS) and gave me `pars = [ 9.940e-01  1.003e+00  1.370e+01]` with associated loss `0.019214857`. Somehow, my gradient descent gave a smaller loss than scipy (which is definitely a coincidence but also nice because it means my optimizer is indeed an optimizer). Check out my plots!

In [ ]:
fn = 'ps04_peak.pkl'
xs, ys = pickle.load(open(fn, 'rb'))

In [ ]:
def predict(pars, xs):
    return pars[0] + pars[1]*jnp.exp(0 - 0.5*(xs - pars[2])**2)

In [ ]:
xplot = np.arange(np.min(xs), np.max(xs), 0.01)
okguess = jnp.array([1, 1.1, 13.3])
plt.title("Pickle file data and model from 'ok' guess")
plt.xlabel("x")
plt.ylabel("y")
plt.plot(xs, ys, "ks", label="data")
plt.plot(xplot, predict(okguess, xplot), "r-", label="model")
plt.ylim(0.8, 2.2)
plt.legend(loc="upper left")
plt.show()

In [ ]:
def loss(pars, xs, ys):
    return jnp.sum((ys - predict(pars, xs))**2)

In [ ]:
print(loss(okguess, xs, ys))

In [ ]:
def descend(f, g, guess, xs, ys):
    tol = 1e-4
    pars = guess
    grad = jnp.ones(pars.shape)
    while (np.linalg.norm(grad) > tol):
        grad = jax.grad(f)(pars, xs, ys)
        pars = pars - g*(grad)
    return pars

In [ ]:
descopt = descend(loss, 0.01, okguess, xs, ys)
print(descopt)
print(loss(descopt, xs, ys))

In [ ]:
scipopt = optimize.minimize(lambda pars: loss(pars, *(xs, ys)), okguess, jac=lambda pars: np.array(
        jax.grad(loss)(jnp.array(pars), xs, ys)))
print(scipopt)
print(loss(scipopt.x, xs, ys))

In [ ]:
plt.title("Pickle file data and model from my gradient descent")
plt.xlabel("x")
plt.ylabel("y")
plt.plot(xs, ys, "ks", label="data")
plt.plot(xplot, predict(descopt, xplot), "r-", label="model")
plt.ylim(0.8, 2.2)
plt.legend(loc="upper left")
plt.show()

In [ ]:
plt.title("Pickle file data and model from scipy L-BFGS")
plt.xlabel("x")
plt.ylabel("y")
plt.plot(xs, ys, "ks", label="data")
plt.plot(xplot, predict(scipopt.x, xplot), "r-", label="model")
plt.ylim(0.8, 2.2)
plt.legend(loc="upper left")
plt.show()

### Problem 2.

_(a)_ Plot!

_(b)_ Plot!

_(c)_ Plot!

In [ ]:
def dft(ys):
    ns = np.arange(len(ys))
    return np.exp(0 - 1j*2*np.pi * np.outer(ns, ns) / len(ys))@ys

In [ ]:
def squarewave(x):
    return 2*(np.floor(2*x) % 2) - 1

In [ ]:
def sawtooth(x):
    return x

In [ ]:
def modsine(x):
    return np.sin(np.pi*x)*np.sin(20*np.pi*x)

In [ ]:
def hoggrange(a, b, N):
    step = (b - a) / N
    return np.arange(a + 0.5*step, b, step)
    #equivalently return step * (np.arange(N) + 0.5) + a

In [ ]:
def fourierplot(f, a, b, N, label):
    xs = hoggrange(a, b, N)
    ks = np.arange(N)
    ys = f(xs)
    cs = np.abs(dft(ys))
    fig, axs = plt.subplots(2, 1, figsize=(5, 6))
    fig.suptitle(label+" in real and frequency space")
    axs[0].plot(xs, ys, "-k")
    axs[0].set_xlabel(r"$x$ (real space)")
    axs[0].set_ylabel(r"$f(x)$ "+"("+label+")")
    axs[1].plot(ks[:np.int32(N/2)], cs[:np.int32(N/2)], "-r", lw=0.5)
    axs[1].set_xlabel(r"$k$ (frequency mode number)")
    axs[1].set_ylabel(r"$|c_k|$ (amplitude)")
    plt.tight_layout()
    plt.show()

In [ ]:
fourierplot(squarewave, -0.5, 0.5, 1000, "Square wave")

In [ ]:
fourierplot(sawtooth, -0.5, 0.5, 1000, "Sawtooth")

In [ ]:
fourierplot(modsine, -1, 1, 1000, "Modulated Sine Wave")

### Problem 3.

_(a)_ Plot! It looks like roughly 4 periods per 500 months, so (roughly) a 125 month cycle (or decade-ish).

_(b)_ Plot!

_(c)_ It's `k=23`! This corresponds to `136.65217391304347` months, which is pretty close to the rough estimate!

_(d)_ It's `458.984375`Hz! 

In [ ]:
fn1 = "sunspots.txt"
data1 = np.loadtxt(fn1)

In [ ]:
plt.plot(data1[:,0], data1[:,1], "-k")
plt.xlabel(r"$t$ (months since Jan 1749)")
plt.ylabel(r"$n$ (observed number of sunspots)")
plt.title("Observed number of sunspots over time")
plt.tight_layout()
plt.show()

In [ ]:
pspec = np.abs(dft(data1[:,1]))**2

In [ ]:
plt.plot(data1[:np.int32(data1.shape[0]/2),0], pspec[:np.int32(data1.shape[0]/2)], "-r")
plt.xlabel(r"$k$ (frequency mode number)")
plt.ylabel(r"$|c_k|^2$ (power spectrum)")
plt.title("Power spectrum of observed number of sunspots")
plt.tight_layout()
plt.show()

In [ ]:
kmax = np.argmax(pspec[1:])
print(kmax)
print(data1.shape[0] / kmax)

In [ ]:
fn2 = "signal.txt"
data2 = np.loadtxt(fn2)
cs2 = np.abs(dft(data2))

In [ ]:
ns = np.arange(data2.shape[0])
plt.plot(ns*1e-4, data2, "-k")
plt.xlabel(r"$t$ (time in s)")
plt.ylabel(r"$s$ (signal)")
plt.title("Signal")
#plt.axvline(0.0216, color="r")
#plt.axvline(0.0238, color="r")
plt.tight_layout()
plt.show()

In [ ]:
plt.plot(ns[:np.int32(data2.shape[0]/2)], cs2[:np.int32(data2.shape[0]/2)], "-r")
plt.xlabel(r"$k$ (frequency mode number)")
plt.ylabel(r"$|c_k|$ (amplitude)")
plt.title("Signal in frequency space")
plt.tight_layout()
plt.show()

In [ ]:
kmax2 = np.argmax(cs2[1:])
print(kmax2)
print(kmax2 / (data2.shape[0] * 1e-4))

### Problem 4.

We do indeed get the powers of 2, though eventually errors emerge:
`1.0
2.0
4.0
8.0
16.0
32.00000000000001
64.00000000000003
128.00000000000003
255.99999999999972
511.99999999998005
1024.0000000000641
2048.0000000000855
4096.000000002532
8192.000000153601
16384.00000006832
32768.00001629912
65535.99988930698
131072.0004021521
262143.89534854155
524287.39244009077
1048613.8007244647`.

In [ ]:
def fstar(z):
    return np.conjugate(np.exp(2*z))

In [ ]:
N = 10000
zs = np.exp(1j*2*np.pi * np.arange(1 + N/2) / N)
fs = fstar(zs)
invs = np.fft.irfft(fs)
for m in range(21):
    print(factorial(m)*invs[m])

### Credits
Big thank yous to the numpy, scipy, and jax documentation. Shoutout Nana Miller for showing me her plots. ChatGPT helped me figure out a bug in question 4 and in my gradient descent.